# 00 - Configuration and data inventory

**Purpose**: find where the data are (locally and on BDAP), check what they contain
(grid, period, calendar, units) and write a configuration file that all the other notebooks read.

This notebook corrects nothing. It only reads and reports problems.

**Do this once**: replace every `TO_FILL` (data roots and file patterns), first locally,
then after exploring BDAP (STAC catalogue and `/eos/jeodpp/data`).

**This is the only place where paths are entered.** Notebooks 01 to 06 read the configuration file written here.

Overall pipeline:
1. **F1** (global): analogue-based downscaling, EC-Earth3 historical (native resolution) -> ERA5 0.25 deg,
   applied to all GCM runs (no WDF / QDM, no regridding).
2. **F2** (RCM domain, no hand-made box): WDF + QDM, RCM 2025 no-hosing -> target GCM_2025_nohosing ERA5-like 0.25 deg,
   then applied unchanged to all RCM runs.

**Running on BDAP**: use JEO-lab (JupyterLab, Python 3, conda environment). The code uses only `pathlib` and forward slashes,
so it runs unchanged on Linux and on Windows.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
from pathlib import Path
import glob
import json
import os
import warnings

import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------------------------------------------------------
# 1. Where is this notebook running? (BDAP = Linux with the EOS storage mounted)
# ----------------------------------------------------------------------------
ON_BDAP = Path("/eos/jeodpp").exists()
PROFILE = "bdap" if ON_BDAP else "local"
print("Active profile:", PROFILE)

# ----------------------------------------------------------------------------
# 2. Roots (TO FILL). Always use "/" in paths, never a backslash.
#    "data" = where the data are read from, "out" = where results are written.
# ----------------------------------------------------------------------------
ROOTS = {
    "local": {
        "data": "TO_FILL/local/path/to/the/data",
        "out": "./outputs",
    },
    "bdap": {
        "data": "/eos/jeodpp/data/projects/DROUGHTOBS",
        "out": "/eos/jeodpp/data/projects/DROUGHTOBS/elisa/GCM_downscaling/outputs",
    },
}
DATA_ROOT = ROOTS[PROFILE]["data"]
OUT_ROOT = Path(ROOTS[PROFILE]["out"])
CONFIG_PATH = OUT_ROOT / f"config_{PROFILE}.json"

## Python environment check

Lists the packages needed by the pipeline. On BDAP, install any missing package in your conda environment.

In [ ]:
import sys
if "." not in sys.path:
    sys.path.insert(0, ".")
try:
    import ba_utils as bu          # ba_utils.py must sit next to this notebook
    missing = bu.check_environment()
except ImportError:
    print("ba_utils.py not found next to this notebook (copy it into the same folder).")

## The datasets

For each dataset, `glob` is a file pattern. `{DATA}` is replaced by the data root and `{var}` by the variable name.
Example patterns: `{DATA}/era5/{var}/*.nc` or `{DATA}/ecearth3/historical/{var}_day_*.nc`.

You can use a different pattern locally and on BDAP: put them in `glob_local` and `glob_bdap`.
A variable can have its own pattern (e.g. `glob_bdap_pr`), and a pattern can be a **list**: the files of all
the patterns are combined (e.g. northern + southern hemisphere files, glued along latitude).

As long as a pattern contains `TO_FILL`, the inventory skips it and says so.

In [ ]:
# Variables handled by the whole pipeline (downscaling + WDF / QDM bias adjustment)
VARIABLES = ["pr", "tasmax", "tasmin"]
# Extra variables: read, inventoried and explored wherever they exist, not (yet) bias-adjusted with WDF + QDM.
#   rsds (surface downwelling shortwave radiation, W m-2): ERA5 only from 2005, so the overlap with
#   EC-Earth3 historical (ends in 2014) is 2005-2014; location in the hosing runs still to be found.
EXTRA_VARIABLES = ["rsds"]
ALL_VARIABLES = VARIABLES + EXTRA_VARIABLES

# Possible variable names in the files (ERA5 and CMIP6 do not use the same ones)
ALIASES = {
    "pr":     ["pr", "tp", "precip", "precipitation"],
    "tasmax": ["tasmax", "mx2t", "tmax", "t2m_max"],
    "tasmin": ["tasmin", "mn2t", "tmin", "t2m_min"],
    "rsds":   ["rsds", "ssrd", "msdwswrf"],
}

A = "TO_FILL"
DATASETS = {
    # ---- observed reference --------------------------------------------------
    "era5": {
        "glob_local": f"{{DATA}}/{A}/era5/{{var}}/*.nc",
        "glob_bdap":  f"{{DATA}}/elisa/GCM_downscaling/Input_data/ERA5_daily/{{var}}_ERA5_day_????.nc",   # finished years only (not *.tmp.nc)
        "domain": "global", "role": "reference",
    },
    # ---- GCM (EC-Earth3) -----------------------------------------------------
    # BDAP: tasmax/tasmin daily global in EC-Earth_hosing/GLOBAL/<run>/ ; the pr files there are MONTHLY,
    # so daily pr is read from the NH (stabilization-*) and SH (day_SH/*) files and glued together.
    "gcm_historical": {
        "glob_local": f"{{DATA}}/{A}/ecearth3_historical/{{var}}/*.nc",
        "glob_bdap":  f"{{DATA}}/elisa/GCM_downscaling/Input_data/EC-Earth3_historical_CMIP6/{{var}}_day_EC-Earth3_historical_r1i1p1f1_*.nc",
        "domain": "global", "role": "gcm_calibration",
    },
    "gcm_2025_nohosing": {
        "glob_local": f"{{DATA}}/{A}/gcm_2025_nohosing/{{var}}/*.nc",
        "glob_bdap":  f"{{DATA}}/EC-Earth_hosing/GLOBAL/2025_Original/{{var}}_*GLOBAL_??????-??????.nc",
        "glob_bdap_pr": [f"{{DATA}}/EC-Earth_hosing/stabilization-ssp585-2025_Original/pr_day_*_NH_??????-??????.nc",    # daily pr: northern
                         f"{{DATA}}/EC-Earth_hosing/day_SH/stabilization-ssp585-2025/pr_day_*_SH_??????-??????.nc"],  # + southern hemisphere
        "glob_bdap_rsds": "TO_FILL: rsds of the hosing runs (location to be found)",
        "domain": "global", "role": "gcm_target_climate",
    },
    "gcm_2025_hosing": {
        "glob_local": f"{{DATA}}/{A}/gcm_2025_hosing/{{var}}/*.nc",
        "glob_bdap":  f"{{DATA}}/EC-Earth_hosing/GLOBAL/2025_Hosing/{{var}}_*GLOBAL_??????-??????.nc",
        "glob_bdap_pr": [f"{{DATA}}/EC-Earth_hosing/stabilization-ssp585-2025_Hosing/pr_day_*_NH_??????-??????.nc",    # daily pr: northern
                         f"{{DATA}}/EC-Earth_hosing/day_SH/stabilization-ssp585-2025_Hosing/pr_day_*_SH_??????-??????.nc"],  # + southern hemisphere
        "glob_bdap_rsds": "TO_FILL: rsds of the hosing runs (location to be found)",
        "domain": "global", "role": "gcm_apply",
    },
    "gcm_2050_nohosing": {
        "glob_local": f"{{DATA}}/{A}/gcm_2050_nohosing/{{var}}/*.nc",
        "glob_bdap":  f"{{DATA}}/EC-Earth_hosing/GLOBAL/2050_Original/{{var}}_*GLOBAL_??????-??????.nc",
        "glob_bdap_pr": [f"{{DATA}}/EC-Earth_hosing/stabilization-ssp585-2050_Original/pr_day_*_NH_??????-??????.nc",    # daily pr: northern
                         f"{{DATA}}/EC-Earth_hosing/day_SH/stabilization-ssp585-2050/pr_day_*_SH_??????-??????.nc"],  # + southern hemisphere
        "glob_bdap_rsds": "TO_FILL: rsds of the hosing runs (location to be found)",
        "domain": "global", "role": "gcm_apply",
    },
    "gcm_2050_hosing": {
        "glob_local": f"{{DATA}}/{A}/gcm_2050_hosing/{{var}}/*.nc",
        "glob_bdap":  f"{{DATA}}/EC-Earth_hosing/GLOBAL/2050_Hosing/{{var}}_*GLOBAL_??????-??????.nc",
        "glob_bdap_pr": [f"{{DATA}}/EC-Earth_hosing/stabilization-ssp585-2050_Hosing/pr_day_*_NH_??????-??????.nc",    # daily pr: northern
                         f"{{DATA}}/EC-Earth_hosing/day_SH/stabilization-ssp585-2050_Hosing/pr_day_*_SH_??????-??????.nc"],  # + southern hemisphere
        "glob_bdap_rsds": "TO_FILL: rsds of the hosing runs (location to be found)",
        "domain": "global", "role": "gcm_apply",
    },
    # ---- RCM (Europe, 0.25 deg) ----------------------------------------------
    "rcm_2025_nohosing": {
        "glob_local": f"{{DATA}}/{A}/rcm_2025_nohosing/{{var}}/*.nc",
        "glob_bdap":  f"{{DATA}}/{A}/rcm_2025_nohosing/{{var}}/*.nc",
        "domain": "europe", "role": "rcm_calibration",
    },
    "rcm_2025_hosing": {
        "glob_local": f"{{DATA}}/{A}/rcm_2025_hosing/{{var}}/*.nc",
        "glob_bdap":  f"{{DATA}}/{A}/rcm_2025_hosing/{{var}}/*.nc",
        "domain": "europe", "role": "rcm_apply",
    },
    "rcm_2050_nohosing": {
        "glob_local": f"{{DATA}}/{A}/rcm_2050_nohosing/{{var}}/*.nc",
        "glob_bdap":  f"{{DATA}}/{A}/rcm_2050_nohosing/{{var}}/*.nc",
        "domain": "europe", "role": "rcm_apply",
    },
    "rcm_2050_hosing": {
        "glob_local": f"{{DATA}}/{A}/rcm_2050_hosing/{{var}}/*.nc",
        "glob_bdap":  f"{{DATA}}/{A}/rcm_2050_hosing/{{var}}/*.nc",
        "domain": "europe", "role": "rcm_apply",
    },
}

## Inventory functions

Files are opened lazily (`xarray` + `dask`): nothing is loaded into memory.

In [ ]:
def resolve_pattern(spec, var):
    """File pattern(s) for the active profile, with {DATA} and {var} replaced.
    `glob_<profile>_<var>` overrides `glob_<profile>`; a list of patterns is returned as a list."""
    pattern = spec.get(f"glob_{PROFILE}_{var}", spec[f"glob_{PROFILE}"])
    fill = lambda p: p.replace("{DATA}", str(DATA_ROOT)).replace("{var}", var)
    return [fill(p) for p in pattern] if isinstance(pattern, list) else fill(pattern)


def as_list(pattern):
    return pattern if isinstance(pattern, list) else [pattern]


def find_var(ds, var):
    """Find the real variable name in the file from the aliases."""
    for name in ALIASES.get(var, [var]):
        if name in ds.data_vars:
            return name
    return None


def find_coord(ds, options):
    for name in options:
        if name in ds.coords or name in ds.dims:
            return name
    return None


def unit_hint(var, units):
    """Conversion needed to obtain mm/day or degrees Celsius."""
    u = (units or "").strip().lower().replace(" ", "")
    if var == "pr":
        if u in ("kgm-2s-1", "kgm**-2s**-1", "kg/m2/s", "mm/s", "kgm^-2s^-1"):
            return "x 86400 -> mm/day"
        if u in ("m", "m/day", "md-1"):
            return "x 1000 -> mm/day"
        if u in ("mm/day", "mmday-1", "mmd-1", "mm"):
            return "ok (mm/day)"
        return "check units"
    if var in ("tasmax", "tasmin"):
        if u in ("k", "kelvin"):
            return "- 273.15 -> degC"
        if u in ("degc", "c", "celsius", "degreec", "degrees_c"):
            return "ok (degC)"
        return "check units"
    if var == "rsds":
        if u in ("wm-2", "w/m2", "wm**-2", "w/m^2", "wm^-2"):
            return "ok (W m-2)"
        if u in ("jm-2", "j/m2", "jm**-2", "jm^-2"):
            return "J m-2 (accumulated): divide by the accumulation period in s"
        return "check units"
    return ""


def inventory_one(name, spec, var):
    pattern = resolve_pattern(spec, var)
    row = {"dataset": name, "var": var, "n_files": 0, "status": "", "pattern": " + ".join(as_list(pattern))}
    if any("TO_FILL" in p for p in as_list(pattern)):
        row["status"] = "to fill in"
        return row
    files = sorted(f for p in as_list(pattern) for f in glob.glob(p))
    row["n_files"] = len(files)
    if not files:
        row["status"] = "no file found"
        return row
    try:
        ds = xr.open_mfdataset(files, combine="by_coords", chunks={}, use_cftime=True)
    except Exception as exc:
        row["status"] = f"error while opening: {exc}"
        return row

    flags = []
    vname = find_var(ds, var)
    if vname is None:
        row["status"] = f"variable missing (looked for: {ALIASES.get(var, [var])}; present: {list(ds.data_vars)})"
        return row
    da = ds[vname]
    row["name_in_file"] = vname
    row["units"] = da.attrs.get("units", "?")
    row["unit_hint"] = unit_hint(var, da.attrs.get("units"))
    if "check" in row["unit_hint"]:
        flags.append("units")

    latn = find_coord(ds, ["lat", "latitude", "y"])
    lonn = find_coord(ds, ["lon", "longitude", "x"])
    if latn and lonn and ds[latn].ndim == 1 and ds[lonn].ndim == 1:
        lat, lon = ds[latn].values, ds[lonn].values
        row["n_lat"], row["n_lon"] = lat.size, lon.size
        row["res_lat"] = round(float(np.median(np.abs(np.diff(lat)))), 4) if lat.size > 1 else None
        row["res_lon"] = round(float(np.median(np.abs(np.diff(lon)))), 4) if lon.size > 1 else None
        row["lat_range"] = f"{lat.min():.2f}..{lat.max():.2f}" + (" (descending)" if lat[0] > lat[-1] else "")
        row["lon_range"] = f"{lon.min():.2f}..{lon.max():.2f}"
        if lon.max() > 180:
            flags.append("lon 0-360")
        if lat[0] > lat[-1]:
            flags.append("descending lat")
    else:
        flags.append("irregular grid or unknown lat/lon names")

    if "time" in ds.coords:
        t = ds["time"].values
        row["n_time"] = int(t.size)
        row["t_start"], row["t_end"] = str(t[0])[:10], str(t[-1])[:10]
        try:
            row["calendar"] = ds["time"].dt.calendar
        except Exception:
            row["calendar"] = "?"
        if t.size > 1:
            steps = np.array([(b - a).total_seconds() / 86400 for a, b in zip(t[:-1], t[1:])])
            med = float(np.median(steps))
            row["step_days"] = round(med, 4)
            n_gap = int((steps > med * 1.5).sum())
            row["n_gaps"] = n_gap
            if abs(med - 1.0) > 1e-6:
                flags.append("time step is not one day")
            if n_gap:
                flags.append(f"{n_gap} gap(s) in time")
            if len(np.unique(t)) != t.size:
                flags.append("duplicated dates")
        if row["calendar"] not in ("standard", "gregorian", "proleptic_gregorian", "?"):
            flags.append(f"calendar {row['calendar']}")
    else:
        flags.append("no time coordinate")

    row["size_gb"] = round(sum(os.path.getsize(f) for f in files) / 1e9, 3)
    row["status"] = "ok" if not flags else "ok, check: " + "; ".join(flags)
    ds.close()
    return row

## Run the inventory

The resulting table is saved as `inventory_<profile>.csv`. Rows saying "to fill in" are normal
as long as the paths are not filled in.

In [ ]:
rows = []
for name, spec in DATASETS.items():
    for var in ALL_VARIABLES:
        rows.append(inventory_one(name, spec, var))

inventory = pd.DataFrame(rows)
cols = ["dataset", "var", "status", "n_files", "name_in_file", "units", "unit_hint",
        "res_lat", "res_lon", "lat_range", "lon_range", "t_start", "t_end", "n_time",
        "calendar", "step_days", "n_gaps", "size_gb", "pattern"]
inventory = inventory.reindex(columns=cols)

OUT_ROOT.mkdir(parents=True, exist_ok=True)
inventory.to_csv(OUT_ROOT / f"inventory_{PROFILE}.csv", index=False)
with pd.option_context("display.max_columns", None, "display.width", 220, "display.max_colwidth", 60):
    display(inventory.drop(columns=["pattern"]))

## Cross-checks

What must be consistent between datasets for the pipeline to work:
- the **calendar** (EC-Earth3 may be `noleap` or proleptic Gregorian, ERA5 is standard);
- the **longitude convention** (0-360 or -180-180);
- the **units** (precipitation in mm/day, temperature in degC);
- overlapping periods (EC-Earth3 historical and ERA5 over 1981-2014).

`ba_utils.read_raw` converts calendars, units and coordinates automatically, but you should know what is being converted.

In [ ]:
ok = inventory[inventory["status"].fillna("").str.startswith("ok")]
if ok.empty:
    print("No readable dataset yet: fill in the TO_FILL paths and run again.")
else:
    print("Calendars found:", sorted(ok["calendar"].dropna().unique()))
    print("Units found per variable:")
    print(ok.groupby("var")["units"].unique().to_string())
    print()
    for var in VARIABLES:
        sub = ok[ok["var"] == var]
        for dsname in ("gcm_historical", "era5"):
            r = sub[sub["dataset"] == dsname]
            if not r.empty:
                print(f"{var:7s} {dsname:15s} {r.iloc[0]['t_start']} -> {r.iloc[0]['t_end']}")
    cals = set(ok["calendar"].dropna().unique())
    if len(cals) > 1:
        print("\nWARNING: different calendars between datasets:", sorted(cals))
        print("  They are converted to 'noleap' (29 February removed) when read with ba_utils.read_raw.")
    lon360 = ok[ok["lon_range"].fillna("").str.contains(r"\.\.(?:1[89]\d|2\d\d|3\d\d)\.")]
    if not lon360.empty and len(lon360) < len(ok):
        print("\nWARNING: mixed longitude conventions (0-360 and -180-180).")
        print("  ba_utils.read_raw puts everything in -180..180.")
    problems = ok[ok["status"].str.contains("check")]
    if not problems.empty:
        print("\nRows to check:")
        for _, r in problems.iterrows():
            print(f"  - {r['dataset']} / {r['var']} : {r['status']}")

## Write the configuration

`config_<profile>.json` holds the resolved patterns, the aliases and the unit conversions.
The following notebooks read it instead of redefining paths.

In [ ]:
config = {
    "profile": PROFILE,
    "data_root": str(DATA_ROOT),
    "out_root": str(OUT_ROOT),
    "variables": VARIABLES,
    "extra_variables": EXTRA_VARIABLES,
    "aliases": ALIASES,
    "datasets": {
        name: {
            "domain": spec["domain"],
            "role": spec["role"],
            "patterns": {var: resolve_pattern(spec, var) for var in ALL_VARIABLES},
        }
        for name, spec in DATASETS.items()
    },
    "unit_hints": {
        f"{r.dataset}/{r.var}": r.unit_hint
        for r in inventory.itertuples() if isinstance(r.unit_hint, str) and r.unit_hint
    },
}
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print("Configuration written:", CONFIG_PATH)

## Next steps

1. Fill in the `TO_FILL` cells, run again, fix what the inventory reports (units, calendar, longitude).
2. Exploration of the raw data, in `001_exploration/`: `01_exploration_data` (all datasets) and `01b_exploration_GCM`
   (EC-Earth3 historical and hosing runs, LISFLOOD domain). Their outputs go to `001_exploration/outputs/` (not in git).
   Before that: `00b_grid_check` (grids) and `00c_land_mask` (LISFLOOD domain masks).
3. Two alternative pipelines (each in its own folder; they can both be run, outputs go to `<out_root>/plan_A/` and `<out_root>/plan_B/`):
   - **`plan_B/`** (shortcut, analogue downscaling directly on the native GCM): `02_step1_Ecearth3historical_downscaling_F1` -> `03_step2_F1_applied_all_GCM` -> `04_step3_RCM_BA_F2` -> `05_step4_F2_applied_all_RCM` -> `06_validations`.
   - **`plan_A/`** (full chain: regrid to 1 deg, WDF+QDM F1, ERA5 downscaling F2, RCM WDF+QDM F3): `02_regridding_1deg` -> `03_step1_Ecearth3historical_BA_F1` -> `04_step2_F1_applied_all_GCM` -> `05_step3_ERA5_downscaling_F2` -> `06_step4_F2_applied_GCM_1deg_to_0.25deg` -> `07_step5_RCM_BA_F3` -> `08_step6_F3_applied_all_RCM` -> `09_validations`.

Both plans share this notebook, notebook 01 and `ba_utils.py` (in this root folder). Each plan has its own downscaling stub to complete with the supervisor's method (`plan_B/f1_downscaler.py`, `plan_A/f2_downscaler.py`).

The other notebooks read the configuration file written here.

Overall pipeline:
1. **F1** (global): analogue-based downscaling, EC-Earth3 historical (native resolution) -> ERA5 0.25 deg,
   applied to all GCM runs (no WDF / QDM, no regridding).
2. **F2** (RCM domain, no hand-made box): WDF + QDM, RCM 2025 no-hosing -> target GCM_2025_nohosing ERA5-like 0.25 deg,
   then applied unchanged to all RCM runs.

**Running on BDAP**: use JEO-lab (JupyterLab, Python 3, conda environment). The code uses only `pathlib` and forward slashes,
so it runs unchanged on Linux and on Windows.